In [2]:
import sklearn
print(sklearn.__version__)

1.8.0


In [4]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ---------------------------------------------------
# Dataset: House Area (sq.ft) vs Rent (in thousands)
# ---------------------------------------------------

X = np.array([
    [500],
    [750],
    [1000],
    [1200],
    [1500],
    [1800],
    [2000],
    [2200],
    [2500],
    [3000]
])

y = np.array([
    8.5,
    12.0,
    15.5,
    20.1,
    24.0,
    28.5,
    35.2,
    38.0,
    44.5,
    55.0
])

# ---------------------------------------------------
# Split dataset into training and testing sets
# ---------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------
# Create Pipeline:
# 1. Standardize data
# 2. Train SVR model
# ---------------------------------------------------

svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

# ---------------------------------------------------
# Hyperparameter tuning using GridSearchCV
# ---------------------------------------------------

param_grid = {
    'svr__kernel': ['linear', 'rbf'],
    'svr__C': [0.1, 1, 10, 100],
    'svr__epsilon': [0.1, 0.5, 1.0],
    'svr__gamma': ['scale', 0.01, 0.1]
}

grid_search = GridSearchCV(
    estimator=svr_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=0
)

# Train all combinations
grid_search.fit(X_train, y_train)

# ---------------------------------------------------
# Best parameters from Grid Search
# ---------------------------------------------------

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV MSE : {-grid_search.best_score_:.4f}")

# ---------------------------------------------------
# Test the best model
# ---------------------------------------------------

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f"\nTest MSE  : {mse:.4f}")
print(f"Test RMSE : {rmse:.4f}")
print(f"Test R2   : {r2:.4f}")

# ---------------------------------------------------
# Manual SVR Model
# ---------------------------------------------------

print("\n--- Manual SVR with RBF Kernel ---")

manual_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(
        kernel='rbf',
        C=100,
        epsilon=0.1,
        gamma=0.1
    ))
])

manual_svr.fit(X_train, y_train)

y_pred_manual = manual_svr.predict(X_test)

print(
    f"Manual RMSE : "
    f"{mean_squared_error(y_test, y_pred_manual, squared=False):.4f}"
)

print(
    f"Manual R2   : "
    f"{r2_score(y_test, y_pred_manual):.4f}"
)

# ---------------------------------------------------
# Residual Table
# ---------------------------------------------------

print("\n--- Predictions on Test Set ---")

print(f"{'Area':>8} {'Actual':>10} {'Predicted':>12} {'Error':>10}")

print("-" * 45)

for xi, ya, yp in zip(X_test.ravel(), y_test, y_pred_manual):

    print(
        f"{xi:>8.0f} "
        f"{ya:>10.2f} "
        f"{yp:>12.2f} "
        f"{ya - yp:>10.2f}"
    )

# ---------------------------------------------------
# Predict Rent for New House
# ---------------------------------------------------

new_house = np.array([[1600]])

predicted_rent = manual_svr.predict(new_house)

print(
    f"\nPredicted rent for 1600 sq.ft : "
    f"Rs. {predicted_rent[0]:.1f}k"
)

Best Parameters: {'svr__C': 100, 'svr__epsilon': 1.0, 'svr__gamma': 'scale', 'svr__kernel': 'linear'}
Best CV MSE : 2.9715


TypeError: got an unexpected keyword argument 'squared'